# RQ2 temporal pair ablation — Part 2

Train only the frozen combined `temporal continuity + stage annealing` branch, then produce the seven-way U/R/SW/RG/ablation comparison. One T4 is sufficient.

## Inputs

Attach the completed Part-1 output containing `e2e_pairwise_pilot_v2`, CIFAR-100, Gate-A `gate_a_summary.json`, and Kaggle secret `github_token`.

In [ ]:
import os,subprocess,sys,json,time,zipfile,importlib
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret('github_token'); assert token
PROJECT_ROOT=Path('/kaggle/working/new-pruning'); askpass=Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n"); askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':token})
try:
    command=['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command,env=env,check=True)
finally: askpass.unlink(missing_ok=True); token=None
os.chdir(PROJECT_ROOT);sys.path.insert(0,str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count()>=1,'Enable a GPU'
GIT_COMMIT=subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip();print(GIT_COMMIT)

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
import rq2_temporal_pair_training as temporal
pilot=importlib.reload(pilot);temporal=importlib.reload(temporal)
INPUT_ROOT=Path('/kaggle/input');DATASET_ROOT=pilot.find_cifar100_root(INPUT_ROOT);GATE_A_SUMMARY=pilot.find_gate_a_summary(INPUT_ROOT)
ROOT=pilot.materialize_progress(INPUT_ROOT,'/kaggle/working/e2e_pairwise_pilot_v2','/kaggle/working/materialized-temporal-part1')
required=[ROOT/'temporal_pair_protocol.json',ROOT/'common_warmup/epoch_010.pt']
for method in ('uniform','resource','pure_sw','resource_geo','sw_continuity','sw_anneal'): required += [ROOT/method/'checkpoints/epoch_100.pt',ROOT/method/'dense_metrics.csv']
missing=[str(p) for p in required if not p.is_file()];assert not missing,f'Incomplete Part-1 input: {missing}'
protocol=temporal.load_temporal_protocol(ROOT);print(json.dumps(protocol,indent=2));print('ROOT:',ROOT)

## Train only the combined frozen method

In [ ]:
started=time.perf_counter()
config=pilot.load_config(ROOT/'resolved_config.yaml',DATASET_ROOT)
final=temporal.train_temporal_branch(config,ROOT,GATE_A_SUMMARY,'sw_continuity_anneal')
print('Final checkpoint:',final);display(__import__('pandas').read_csv(ROOT/'sw_continuity_anneal/temporal_policy_history.csv'))
decision=temporal.summarize_temporal_ablation(ROOT);print(json.dumps(decision,indent=2))
display(__import__('pandas').read_csv(ROOT/'temporal_pair_ablation_summary.csv'))
print(f'Part 2 completed in {(time.perf_counter()-started)/3600:.2f} h')

In [ ]:
required=['temporal_pair_ablation_summary.csv','temporal_pair_ablation_decision.json','sw_continuity_anneal/checkpoints/epoch_100.pt','sw_continuity_anneal/temporal_policy_history.csv']
missing=[name for name in required if not (ROOT/name).is_file()];assert not missing,missing
bundle=Path('/kaggle/working/rq2-temporal-pair-ablation-complete.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path,Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print(bundle,f'{bundle.stat().st_size/2**30:.2f} GiB');bundle